# 9.4 推理框架 (Inference Frameworks)

推理框架将各种优化技术整合为生产级服务，是大模型部署的核心基础设施。选择合适的推理框架直接影响服务延迟、吞吐量和成本。

本节涵盖：
- 推理框架核心架构
- 框架对比与选型
- torch.compile优化
- 推理服务性能基准
- 生产部署架构

## 1. 推理框架核心架构

**目的**：理解推理框架的内部架构，掌握各组件职责

**典型推理服务架构**：
```
Client → API Gateway → Scheduler → Worker Pool → GPU
                        ↓
                   KV Cache Manager
                        ↓
                   Model Runner
```

**核心组件**：
- **Scheduler**：请求调度，决定哪些请求组成batch、何时执行prefill/decode
- **KV Cache Manager**：管理KV缓存的分配和回收，PagedAttention的核心
- **Model Runner**：执行模型前向计算，管理GPU显存
- **API Server**：提供OpenAI兼容API，处理请求/响应格式化

**关键性能指标**：
- **TTFT（Time To First Token）**：从请求到第一个token的延迟，受prefill影响
- **TPOT（Time Per Output Token）**：每个输出token的延迟，受decode速度影响
- **吞吐量**：单位时间处理的token数，受batch size和并行度影响
- **首token延迟P99**：99%请求的首token延迟，衡量服务质量

In [ ]:
import torch
import torch.nn as nn
import time
from collections import deque
import random

torch.manual_seed(42)

class SimpleInferenceScheduler:
    def __init__(self, max_batch_size=8, max_waiting=16):
        self.max_batch_size = max_batch_size
        self.max_waiting = max_waiting
        self.waiting_queue = deque()
        self.running_batch = []
        self.kv_cache_blocks = {}  # request_id -> num_blocks
        self.total_blocks = 100
        self.used_blocks = 0
        self.next_id = 0

    def submit_request(self, prompt_len, max_new_tokens):
        req_id = self.next_id
        self.next_id += 1
        self.waiting_queue.append({
            'id': req_id, 'prompt_len': prompt_len,
            'max_new_tokens': max_new_tokens, 'generated': 0,
            'phase': 'prefill'
        })
        return req_id

    def schedule(self):
        while self.waiting_queue and len(self.running_batch) < self.max_batch_size:
            req = self.waiting_queue[0]
            needed_blocks = (req['prompt_len'] + req['max_new_tokens']) // 16 + 1
            if self.used_blocks + needed_blocks <= self.total_blocks:
                self.waiting_queue.popleft()
                self.running_batch.append(req)
                self.kv_cache_blocks[req['id']] = needed_blocks
                self.used_blocks += needed_blocks
            else:
                break
        return self.running_batch

    def step(self):
        completed = []
        for req in self.running_batch:
            req['generated'] += 1
            if req['generated'] >= req['max_new_tokens']:
                completed.append(req)
        for req in completed:
            self.running_batch.remove(req)
            self.used_blocks -= self.kv_cache_blocks.pop(req['id'])
        return completed

scheduler = SimpleInferenceScheduler(max_batch_size=4, max_waiting=10)

print('=== Inference Scheduler Simulation ===')
print(f'KV Cache: {scheduler.total_blocks} blocks, max_batch={scheduler.max_batch_size}')

for i in range(6):
    prompt_len = random.randint(10, 50)
    max_new = random.randint(20, 80)
    req_id = scheduler.submit_request(prompt_len, max_new)
    print(f'Submitted request {req_id}: prompt={prompt_len}, max_new={max_new}')

total_steps = 0
total_completed = 0
while scheduler.running_batch or scheduler.waiting_queue:
    batch = scheduler.schedule()
    if not batch:
        break
    completed = scheduler.step()
    total_steps += 1
    total_completed += len(completed)
    if completed or total_steps % 20 == 0:
        print(f'Step {total_steps}: running={len(batch)}, waiting={len(scheduler.waiting_queue)}, '
              f'kv_used={scheduler.used_blocks}/{scheduler.total_blocks}, completed={len(completed)}')
    if total_steps > 200:
        break

print(f'\nTotal: {total_steps} steps, {total_completed} requests completed')
print(f'\nKey: Scheduler manages batch composition and KV cache allocation.')
print(f'When KV cache is full, new requests must wait (backpressure).')
print(f'vLLM uses PagedAttention for efficient KV cache block management.')

## 2. 框架对比与选型

**目的**：根据场景选择合适的推理框架

| 维度 | TensorRT-LLM | vLLM | TGI | ONNX Runtime |
|------|-------------|------|-----|-------------|
| **性能** | 最高 | 高 | 高 | 中 |
| **易用性** | 低（需预编译） | 高（pip install） | 中（Docker） | 中 |
| **硬件** | NVIDIA GPU | NVIDIA/AMD GPU | NVIDIA GPU | CPU/GPU/DPU |
| **量化** | FP8/INT4/INT8 | GPTQ/AWQ/FP8 | GPTQ/bitsandbytes | INT8/INT4 |
| **KV Cache** | Paged+量化 | PagedAttention | Paged | 基础 |
| **Batching** | Continuous | Continuous | Continuous | Static |
| **Speculative** | 支持 | 支持 | 支持 | 不支持 |
| **适用场景** | 大规模GPU集群 | 通用GPU推理 | HF生态部署 | 边缘/跨平台 |

**选型建议**：
- **追求极致性能**：TensorRT-LLM（需NVIDIA GPU + 预编译时间）
- **快速部署/开源**：vLLM（最活跃的开源社区，功能最全）
- **HuggingFace生态**：TGI（与HF模型无缝集成）
- **CPU/边缘部署**：ONNX Runtime（跨平台，低资源）

In [ ]:
print('=== Framework Selection Decision Tree ===')
print()
print('1. What hardware?')
print('   ├─ NVIDIA A100/H100 → TensorRT-LLM or vLLM')
print('   ├─ NVIDIA consumer GPU → vLLM')
print('   ├─ AMD GPU → vLLM (ROCm)')
print('   └─ CPU only → ONNX Runtime or llama.cpp')
print()
print('2. What throughput do you need?')
print('   ├─ >10K tokens/sec → TensorRT-LLM (multi-GPU)')
print('   ├─ 1K-10K tokens/sec → vLLM')
print('   └─ <1K tokens/sec → Any framework works')
print()
print('3. What latency SLA?')
print('   ├─ TTFT < 100ms → TensorRT-LLM or vLLM')
print('   ├─ TTFT < 500ms → vLLM or TGI')
print('   └─ TTFT < 2s → Any framework works')
print()
print('4. How quickly do you need to deploy?')
print('   ├─ Today → vLLM (pip install vllm)')
print('   ├─ This week → TGI (docker run)')
print('   └─ Can wait → TensorRT-LLM (best perf after tuning)')

print('\n=== Typical Deployment Commands ===')
print()
print('# vLLM (easiest):')
print('python -m vllm.entrypoints.openai.api_server \\')
    --model meta-llama/Llama-2-7b-hf \\')
    --tensor-parallel-size 2 \\')
    --gpu-memory-utilization 0.9')
print()
print('# TGI (Docker):')
print('docker run --gpus all -p 8080:80 \\')
    ghcr.io/huggingface/text-generation-inference:latest \\')
    --model-id meta-llama/Llama-2-7b-hf')
print()
print('# TensorRT-LLM (build + deploy):')
print('python convert_checkpoint.py --model_dir ./llama-7b --output_dir ./trt_ckpt')
print('trtllm-build --checkpoint_dir ./trt_ckpt --output_dir ./trt_engine \\')
    --gemm_plugin auto --use_paged_context_fmha enable')
print()
print('Key: vLLM is the best starting point for most use cases.')
print('TensorRT-LLM requires more setup but gives the best throughput.')

## 3. torch.compile优化

**目的**：使用PyTorch原生编译优化加速推理

**torch.compile原理**：
- 将PyTorch代码编译为优化的TorchDynamo IR
- 通过后端（Inductor/TensorRT）生成优化kernel
- 支持算子融合、内存布局优化、自动混合精度

**适用场景**：
- 模型结构固定、输入shape变化少
- 不需要KV Cache等高级特性
- 适合小模型推理或模型中部分子模块的优化

In [ ]:
import torch
import torch.nn as nn
import time

torch.manual_seed(42)

class TransformerBlock(nn.Module):
    def __init__(self, d_model=256, n_heads=4):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_model*4), nn.GELU(), nn.Linear(d_model*4, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        x = x + self.attn(self.norm1(x), self.norm1(x), self.norm1(x))[0]
        x = x + self.ffn(self.norm2(x))
        return x

class SmallLM(nn.Module):
    def __init__(self, d_model=256, n_heads=4, n_layers=4, vocab_size=1000):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([TransformerBlock(d_model, n_heads) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, input_ids):
        h = self.embed(input_ids)
        for layer in self.layers:
            h = layer(h)
        return self.head(self.norm(h))

model = SmallLM()
input_ids = torch.randint(0, 1000, (4, 64))

print('=== torch.compile Optimization ===')
print(f'Model params: {sum(p.numel() for p in model.parameters()):,}')

n_warmup = 5
n_runs = 50

for _ in range(n_warmup):
    _ = model(input_ids)

start = time.time()
for _ in range(n_runs):
    _ = model(input_ids)
eager_time = (time.time() - start) / n_runs * 1000

compiled_model = torch.compile(model, mode='reduce-overhead')
for _ in range(n_warmup):
    _ = compiled_model(input_ids)

start = time.time()
for _ in range(n_runs):
    _ = compiled_model(input_ids)
compiled_time = (time.time() - start) / n_runs * 1000

print(f'\nEager mode:    {eager_time:.2f} ms')
print(f'torch.compile: {compiled_time:.2f} ms')
if compiled_time < eager_time:
    print(f'Speedup: {eager_time/compiled_time:.2f}x')

print(f'\ntorch.compile modes:')
print(f'  "reduce-overhead": Best for GPU, uses CUDA graphs')
print(f'  "default": Balanced compilation time vs speedup')
print(f'  "max-autotune": Tries many kernels, longest compile, best speedup')
print(f'\nKey: torch.compile is easy to use (1 line change) but limited for LLM serving.')
print(f'For production LLM serving, use vLLM/TensorRT-LLM for KV cache and batching.')

## 4. 推理性能基准测试

**目的**：量化推理性能，指导优化和容量规划

**基准测试方法论**：
- **隔离变量**：分别测试prefill和decode速度
- **控制batch size**：从1到max逐步增加，找到吞吐量-延迟最优点
- **控制序列长度**：短/中/长序列性能差异大
- **多次运行取平均**：消除GPU频率波动影响

**典型性能数据**（7B模型，A100-80GB，BF16）：
- vLLM：~3000 tokens/sec（batch=32），TTFT~50ms
- TensorRT-LLM：~5000 tokens/sec（batch=32），TTFT~30ms
- 单请求decode：~30ms/token（无batch）

In [ ]:
import torch
import torch.nn as nn
import time

torch.manual_seed(42)

class BenchmarkModel(nn.Module):
    def __init__(self, d_model=256, n_heads=4, n_layers=4, vocab_size=1000):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            nn.ModuleDict({
                'attn': nn.MultiheadAttention(d_model, n_heads, batch_first=True),
                'ffn': nn.Sequential(nn.Linear(d_model, d_model*4), nn.GELU(), nn.Linear(d_model*4, d_model)),
                'norm1': nn.LayerNorm(d_model),
                'norm2': nn.LayerNorm(d_model),
            }) for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, input_ids, past_kv=None, use_cache=False):
        h = self.embed(input_ids)
        new_kv = []
        for i, layer in enumerate(self.layers):
            residual = h
            h = layer['norm1'](h)
            if past_kv and i < len(past_kv):
                h_attn, kv = layer['attn'](h, h, h, need_weights=False)
                new_kv.append(kv)
            else:
                h_attn, kv = layer['attn'](h, h, h, need_weights=False)
                new_kv.append(kv)
            h = residual + h_attn
            h = h + layer['ffn'](layer['norm2'](h))
        return self.head(self.norm(h)), new_kv

model = BenchmarkModel()
model.eval()

print('=== Inference Benchmark ===')
print(f'Model params: {sum(p.numel() for p in model.parameters()):,}')

print(f'\n--- Prefill Latency (batch=1) ---')
for seq_len in [32, 64, 128, 256, 512]:
    input_ids = torch.randint(0, 1000, (1, seq_len))
    for _ in range(3):
        with torch.no_grad():
            _ = model(input_ids)
    start = time.time()
    n_runs = 20
    for _ in range(n_runs):
        with torch.no_grad():
            _ = model(input_ids)
    elapsed = (time.time() - start) / n_runs * 1000
    print(f'  seq_len={seq_len:4d}: {elapsed:.2f} ms ({seq_len/elapsed*1000:.0f} tokens/sec)')

print(f'\n--- Decode Latency (1 token, with KV cache) ---')
prompt = torch.randint(0, 1000, (1, 64))
with torch.no_grad():
    _, kv = model(prompt, use_cache=True)

single_token = torch.randint(0, 1000, (1, 1))
for _ in range(5):
    with torch.no_grad():
        _ = model(single_token)
start = time.time()
n_runs = 100
for _ in range(n_runs):
    with torch.no_grad():
        _ = model(single_token)
decode_time = (time.time() - start) / n_runs * 1000
print(f'  1 token decode: {decode_time:.2f} ms ({1/decode_time*1000:.0f} tokens/sec)')

print(f'\n--- Throughput vs Batch Size ---')
for batch_size in [1, 2, 4, 8, 16]:
    input_ids = torch.randint(0, 1000, (batch_size, 64))
    for _ in range(3):
        with torch.no_grad():
            _ = model(input_ids)
    start = time.time()
    n_runs = 20
    for _ in range(n_runs):
        with torch.no_grad():
            _ = model(input_ids)
    elapsed = (time.time() - start) / n_runs * 1000
    throughput = batch_size * 64 / elapsed * 1000
    print(f'  batch={batch_size:2d}: {elapsed:.2f} ms, throughput={throughput:.0f} tokens/sec')

print(f'\nKey: Prefill is compute-bound (O(n²)); decode is memory-bound (O(n)).')
print(f'Larger batch sizes improve throughput but increase per-request latency.')
print(f'Production systems must balance throughput vs latency SLA.')

## 5. 生产部署架构

**目的**：构建可扩展、高可用的推理服务

**典型生产架构**：
```
                    ┌─────────────┐
                    │  Load       │
                    │  Balancer   │
                    └──────┬──────┘
                           │
              ┌────────────┼────────────┐
              │            │            │
        ┌─────┴────┐ ┌────┴─────┐ ┌────┴─────┐
        │ API      │ │ API      │ │ API      │
        │ Gateway  │ │ Gateway  │ │ Gateway  │
        └─────┬────┘ └────┬─────┘ └────┬─────┘
              │            │            │
        ┌─────┴────┐ ┌────┴─────┐ ┌────┴─────┐
        │ vLLM     │ │ vLLM     │ │ vLLM     │
        │ Worker   │ │ Worker   │ │ Worker   │
        │ (2xGPU)  │ │ (2xGPU)  │ │ (2xGPU)  │
        └──────────┘ └──────────┘ └──────────┘
```

**关键设计决策**：
- **模型并行度**：TP=2需要2GPU/副本，TP=4需要4GPU/副本
- **副本数量**：根据QPS和延迟SLA计算所需副本数
- **自动扩缩容**：基于GPU利用率和请求队列长度
- **模型路由**：不同模型/量化版本路由到不同副本组
- **流式输出**：SSE（Server-Sent Events）实现token级流式响应

In [ ]:
import math

print('=== Production Deployment Capacity Planning ===')

class CapacityPlanner:
    def __init__(self, model_size_gb, gpu_memory_gb, gpu_count,
                 tokens_per_sec, avg_input_len, avg_output_len):
        self.model_size_gb = model_size_gb
        self.gpu_memory_gb = gpu_memory_gb
        self.gpu_count = gpu_count
        self.tokens_per_sec = tokens_per_sec
        self.avg_input_len = avg_input_len
        self.avg_output_len = avg_output_len

    def max_concurrent_requests(self, kv_cache_ratio=0.7):
        available_kv = self.gpu_memory_gb * self.gpu_count * kv_cache_ratio
        kv_per_request = (self.avg_input_len + self.avg_output_len) * 2 * 2 / 1e9 * 4096
        return int(available_kv / kv_per_request)

    def max_qps(self, batch_size=None):
        if batch_size is None:
            batch_size = self.max_concurrent_requests()
        tokens_per_req = self.avg_input_len + self.avg_output_len
        time_per_batch = tokens_per_req * batch_size / self.tokens_per_sec
        return batch_size / time_per_batch

    def replicas_for_qps(self, target_qps):
        single_qps = self.max_qps(batch_size=min(32, self.max_concurrent_requests()))
        return math.ceil(target_qps / single_qps)

print('--- LLaMA-7B on A100-80GB ---')
planner_7b = CapacityPlanner(
    model_size_gb=14, gpu_memory_gb=80, gpu_count=1,
    tokens_per_sec=3000, avg_input_len=500, avg_output_len=200
)
print(f'Max concurrent requests: ~{planner_7b.max_concurrent_requests()}')
print(f'Max QPS (batch=32): ~{planner_7b.max_qps(batch_size=32):.1f}')
print(f'Replicas for 100 QPS: {planner_7b.replicas_for_qps(100)}')

print(f'\n--- LLaMA-70B on 4xA100-80GB (TP=4) ---')
planner_70b = CapacityPlanner(
    model_size_gb=140, gpu_memory_gb=320, gpu_count=4,
    tokens_per_sec=2000, avg_input_len=500, avg_output_len=200
)
print(f'Max concurrent requests: ~{planner_70b.max_concurrent_requests()}')
print(f'Max QPS (batch=32): ~{planner_70b.max_qps(batch_size=32):.1f}')
print(f'Replicas for 100 QPS: {planner_70b.replicas_for_qps(100)}')

print(f'\n--- Cost Analysis (A100 $2/hr on-demand) ---')
cost_7b = planner_7b.replicas_for_qps(100) * 1 * 2
cost_70b = planner_70b.replicas_for_qps(100) * 4 * 2
print(f'7B model @ 100 QPS: {planner_7b.replicas_for_qps(100)} replicas × 1 GPU = ${cost_7b}/hr')
print(f'70B model @ 100 QPS: {planner_70b.replicas_for_qps(100)} replicas × 4 GPUs = ${cost_70b}/hr')

print(f'\nKey: Capacity planning requires understanding throughput, latency, and cost tradeoffs.')
print(f'Quantization (INT4/INT8) can reduce model size by 2-4x, enabling larger batches or fewer GPUs.')
print(f'Auto-scaling based on request queue length is essential for cost efficiency.')